In [ ]:
from pathlib import Path
import shutil, json, csv, random, re
from collections import defaultdict, Counter
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
DATA_INTERIM = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_512_noborder_balanced")
LABEL_STUDIO_JSON = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels_balanced.json")

# Optional extra images: auto-label as non-empty + overlap + non-contraband
EXTRA_NONCONTRABAND_OVERLAP_DIR = Path("../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug")
ENABLE_EXTRA_BALANCING = True

OUT_BASE = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/gray")
OUT_IMAGES = OUT_BASE / "images"
OUT_INDEX = OUT_BASE / "index.csv"
OUT_SPLITS = OUT_BASE / "splits.json"

# Two separate label folders
OUT_SPATIAL_LABELS_DIR = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/gray/spatial_overlap_isolated")
OUT_THREAT_LABELS_DIR  = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2_two_models/gray/threat_contraband_noncontraband")

TRAIN_RATIO = 0.85
VAL_RATIO = 0.10
TEST_RATIO = 0.05
SEED = 42
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

LABEL_KEYS = ["Empty", "Non-Empty", "Overlap", "Isolated", "Contraband", "Non-Contraband"]

EXTRA_LABELS = {
    "Empty": 0,
    "Non-Empty": 1,
    "Overlap": 1,
    "Isolated": 0,
    "Contraband": 0,
    "Non-Contraband": 1,
}

# ============================================================
# HELPERS
# ============================================================
def list_images(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def safe_name(prefix: str, path: Path):
    return f"{prefix}__{path.name}"

def base_group_id(filename: str) -> str:
    # Keeps original + augmented versions together
    stem = Path(filename).stem
    stem = re.sub(r"_(orig|balflip\d+|aug\d+|flip\d+)$", "", stem)
    # Remove source prefix while grouping if added below
    stem = re.sub(r"^(main|extra)__", "", stem)
    return stem

def extract_timestamp(filename: str) -> str:
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})-(\d{3})", filename)
    if not m:
        return ""
    return f"{m.group(1)}T{m.group(2).replace('-', ':')}.{m.group(3)}"

def normalize_label_name(x):
    return str(x).lower().strip().replace("-", "_").replace(" ", "_")

def load_label_studio_labels(json_path: Path):
    data = json.load(open(json_path, "r"))
    labels_by_filename = {}

    for item in data:
        if not isinstance(item, dict):
            continue

        fname = Path(item.get("image", "")).name
        raw_labels = item.get("labels", [])

        labels = {k: 0 for k in LABEL_KEYS}
        norm = {normalize_label_name(x) for x in raw_labels}

        if "empty" in norm:
            labels["Empty"] = 1
        if "non_empty" in norm or "nonempty" in norm:
            labels["Non-Empty"] = 1
        if "overlap" in norm:
            labels["Overlap"] = 1
        if "isolated" in norm:
            labels["Isolated"] = 1
        if "contraband" in norm:
            labels["Contraband"] = 1
        if "non_contraband" in norm or "noncontraband" in norm:
            labels["Non-Contraband"] = 1

        if fname:
            # store normal + prefixed names
            labels_by_filename[fname] = labels
            labels_by_filename[safe_name("main", Path(fname))] = labels

    return labels_by_filename

def count_labels(image_paths, labels_by_filename):
    c = Counter()
    for p in image_paths:
        labels = labels_by_filename.get(p.name)
        if not labels:
            continue
        for k, v in labels.items():
            if v == 1:
                c[k] += 1
    return c

def add_extra_noncontraband_overlap_images(image_paths, labels_by_filename, seed=42):
    if not ENABLE_EXTRA_BALANCING:
        return image_paths, labels_by_filename

    extra_paths = list_images(EXTRA_NONCONTRABAND_OVERLAP_DIR)
    if not extra_paths:
        print("No extra images found or folder missing:", EXTRA_NONCONTRABAND_OVERLAP_DIR)
        return image_paths, labels_by_filename

    counts = count_labels(image_paths, labels_by_filename)

    need_overlap = max(0, counts["Isolated"] - counts["Overlap"])
    need_non_contraband = max(0, counts["Contraband"] - counts["Non-Contraband"])

    # Because extra images add BOTH overlap and non_contraband,
    # add enough to fix the larger deficit, but not more than available.
    num_to_add = min(max(need_overlap, need_non_contraband), len(extra_paths))

    rng = random.Random(seed)
    rng.shuffle(extra_paths)
    selected = extra_paths[:num_to_add]

    print("\nExtra balancing:")
    print("  Before counts:", dict(counts))
    print("  Need overlap:", need_overlap)
    print("  Need non_contraband:", need_non_contraband)
    print("  Extra available:", len(extra_paths))
    print("  Extra selected:", num_to_add)

    # Rename virtually with prefix to avoid filename collision
    copied_extra_paths = []
    tmp_extra_root = OUT_BASE / "_extra_selected_tmp"
    tmp_extra_root.mkdir(parents=True, exist_ok=True)

    for p in selected:
        new_name = safe_name("extra", p)
        dst = tmp_extra_root / new_name
        shutil.copy2(p, dst)
        labels_by_filename[new_name] = dict(EXTRA_LABELS)
        copied_extra_paths.append(dst)

    return image_paths + copied_extra_paths, labels_by_filename

def split_grouped_images(image_paths, seed=42):
    groups = defaultdict(list)
    for p in image_paths:
        groups[base_group_id(p.name)].append(p)

    group_ids = sorted(groups.keys())
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n = len(group_ids)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    train_g = set(group_ids[:n_train])
    val_g = set(group_ids[n_train:n_train + n_val])
    test_g = set(group_ids[n_train + n_val:])

    rows = []
    for gid, plist in groups.items():
        split = "train" if gid in train_g else "val" if gid in val_g else "test"
        for p in plist:
            rows.append({
                "split": split,
                "src_path": str(p),
                "filename": p.name,
                "group_id": gid,
                "scan_timestamp": extract_timestamp(p.name),
            })
    return rows

def get_labels(labels_by_filename, filename):
    labels = labels_by_filename.get(filename)
    if labels is None:
        return {k: 0 for k in LABEL_KEYS}
    return labels

def valid_spatial(labels):
    return (labels["Overlap"], labels["Isolated"]) in [(1, 0), (0, 1)]

def valid_threat(labels):
    return (labels["Contraband"], labels["Non-Contraband"]) in [(1, 0), (0, 1)]

def write_outputs(rows, labels_by_filename):
    for d in [OUT_BASE, OUT_IMAGES, OUT_SPATIAL_LABELS_DIR, OUT_THREAT_LABELS_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    for sp in ("train", "val", "test"):
        (OUT_IMAGES / sp).mkdir(parents=True, exist_ok=True)

    final_rows = []
    splits_json = {"train": [], "val": [], "test": []}
    spatial_items = {"train": [], "val": [], "test": []}
    threat_items = {"train": [], "val": [], "test": []}
    missing = []

    for r in rows:
        split = r["split"]
        src = Path(r["src_path"])

        # Prefix main images to avoid collision with extra
        if not src.name.startswith("extra__"):
            new_name = safe_name("main", src)
        else:
            new_name = src.name

        dst_rel = f"images/{split}/{new_name}"
        dst_abs = OUT_BASE / dst_rel
        shutil.copy2(src, dst_abs)

        labels = get_labels(labels_by_filename, new_name)
        if labels == {k: 0 for k in LABEL_KEYS}:
            missing.append(new_name)

        final_rows.append({
            "filepath": dst_rel,
            "split": split,
            "filename": new_name,
            "scan_timestamp": r["scan_timestamp"],
            "source_interim_path": str(src),
            "group_id": r["group_id"],
            "empty": labels["Empty"],
            "non_empty": labels["Non-Empty"],
            "overlap": labels["Overlap"],
            "isolated": labels["Isolated"],
            "contraband": labels["Contraband"],
            "non_contraband": labels["Non-Contraband"],
        })

        splits_json[split].append(dst_rel)

        if valid_spatial(labels):
            spatial_items[split].append({
                "image": dst_rel,
                "label": "overlap" if labels["Overlap"] == 1 else "isolated",
                "class_id": 1 if labels["Overlap"] == 1 else 0,
                "overlap": labels["Overlap"],
                "isolated": labels["Isolated"],
            })

        if valid_threat(labels):
            threat_items[split].append({
                "image": dst_rel,
                "label": "contraband" if labels["Contraband"] == 1 else "non_contraband",
                "class_id": 1 if labels["Contraband"] == 1 else 0,
                "contraband": labels["Contraband"],
                "non_contraband": labels["Non-Contraband"],
            })

    with open(OUT_INDEX, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
        w.writeheader()
        w.writerows(final_rows)

    with open(OUT_SPLITS, "w") as f:
        json.dump({
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "source_images": str(DATA_INTERIM),
            "source_labels": str(LABEL_STUDIO_JSON),
            "extra_noncontraband_overlap_dir": str(EXTRA_NONCONTRABAND_OVERLAP_DIR),
            "seed": SEED,
            "ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
            "counts": {k: len(v) for k, v in splits_json.items()},
            "missing_label_count": len(missing),
            "missing_labels": missing[:50],
            "splits": splits_json,
        }, f, indent=2)

    for split in ("train", "val", "test"):
        json.dump(spatial_items[split], open(OUT_SPATIAL_LABELS_DIR / f"{split}.json", "w"), indent=2)
        json.dump(threat_items[split], open(OUT_THREAT_LABELS_DIR / f"{split}.json", "w"), indent=2)

    return final_rows, spatial_items, threat_items, missing

def print_distribution(name, items):
    print(f"\n{name}")
    for split in ("train", "val", "test"):
        cnt = Counter([x["label"] for x in items[split]])
        print(f"  {split}: total={len(items[split])}, {dict(cnt)}")

# ============================================================
# RUN
# ============================================================
random.seed(SEED)

if OUT_BASE.exists():
    print("Removing old output:", OUT_BASE)
    shutil.rmtree(OUT_BASE)

print("Loading labels...")
labels_by_filename = load_label_studio_labels(LABEL_STUDIO_JSON)
print("Loaded labels:", len(labels_by_filename))

print("Listing main images...")
main_image_paths = list_images(DATA_INTERIM)
print("Main images:", len(main_image_paths))

# Prefix main labels for later copy names
for p in main_image_paths:
    if p.name in labels_by_filename:
        labels_by_filename[safe_name("main", p)] = labels_by_filename[p.name]

image_paths, labels_by_filename = add_extra_noncontraband_overlap_images(
    main_image_paths,
    labels_by_filename,
    seed=SEED,
)

print("Images after optional extra balancing:", len(image_paths))

rows = split_grouped_images(image_paths, seed=SEED)
final_rows, spatial_items, threat_items, missing = write_outputs(rows, labels_by_filename)

print("\nSaved processed dataset to:", OUT_BASE)
print("Index saved to:", OUT_INDEX)
print("Splits saved to:", OUT_SPLITS)
print("Spatial labels:", OUT_SPATIAL_LABELS_DIR)
print("Threat labels:", OUT_THREAT_LABELS_DIR)
print("Missing label count:", len(missing))

print_distribution("Spatial model labels: 0=isolated, 1=overlap", spatial_items)
print_distribution("Threat model labels: 0=non_contraband, 1=contraband", threat_items)
